# 🎙️ XTTS v2 Vietnamese Fine-Tuning - ALL 5 BUGS FIXED

**Fixed bugs:**
- ✅ Bug #1: W&B hanging (disabled before imports)
- ✅ Bug #2: Audio path mismatch (auto-detected and fixed)
- ✅ Bug #3: audio_root_remap properly configured
- ✅ Bug #4: T4 OOM prevention (batch_size=2, FP16, etc.)
- ✅ Bug #5: Reference audio auto-selected

## Setup Requirements:

### 1. Kaggle Settings:
- **Accelerator**: GPU T4 x1
- **Internet**: ON
- **Persistence**: Files only

### 2. Add Datasets:
- `tinthnhphm21022004/data-speech-to-text`
- `thanhphamtien2102224/weight-phowhisper`

### 3. Run:
- Click **Run All**
- Training time: ~2-3 hours

---

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# [STEP 1/9] FIX BUG #1: Disable W&B BEFORE any imports
# ══════════════════════════════════════════════════════════════════════════════
import os
os.environ['WANDB_MODE'] = 'disabled'
os.environ['WANDB_DISABLED'] = 'true'
os.environ['WANDB_SILENT'] = 'true'

print('='*80)
print('🚀 XTTS v2 Vietnamese Fine-Tuning - ALL 5 BUGS FIXED')
print('='*80)
print('✅ [BUG #1 FIXED] W&B disabled')

import sys
import json
import shutil
import subprocess
import importlib
import gc
from pathlib import Path

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# [STEP 2/9] Check GPU
# ══════════════════════════════════════════════════════════════════════════════
print('\n[STEP 2/9] Checking GPU...')
import torch

print(f'Python: {sys.version.split()[0]}')
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')

if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    vram_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f'VRAM: {vram_gb:.1f} GB')
    if vram_gb < 15:
        print('⚠️  Warning: Less than 16GB VRAM detected')
else:
    print('❌ No GPU detected! Training will fail.')
    raise RuntimeError('GPU required for training')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# [STEP 3/9] Clone repository
# ══════════════════════════════════════════════════════════════════════════════
print('\n[STEP 3/9] Cloning repository...')
REPO_DIR = '/kaggle/working/finetuneXTTSv2'

if os.path.isdir(REPO_DIR):
    shutil.rmtree(REPO_DIR)
    print('🗑️  Removed old repository')

# Clear cached modules
stale = [k for k in sys.modules if k.startswith('xtts_finetune')]
for k in stale:
    del sys.modules[k]

subprocess.check_call(
    ['git', 'clone', 'https://github.com/thanhptks212k4/finetuneXTTSv2.git', REPO_DIR],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)
print(f'✅ Repository cloned to {REPO_DIR}')

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
importlib.invalidate_caches()

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# [STEP 4/9] Install dependencies
# ══════════════════════════════════════════════════════════════════════════════
print('\n[STEP 4/9] Installing dependencies...')

subprocess.check_call(
    [sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', 'pip'],
    stdout=subprocess.DEVNULL,
)

subprocess.check_call(
    [sys.executable, '-m', 'pip', 'install', '-q', 
     'git+https://github.com/idiap/coqui-ai-TTS.git'],
    stdout=subprocess.DEVNULL,
)

subprocess.check_call(
    [sys.executable, '-m', 'pip', 'install', '-q', 
     'huggingface_hub', 'librosa', 'soundfile', 'torchaudio'],
    stdout=subprocess.DEVNULL,
)

print('✅ All dependencies installed')

# Verify installations
import numpy as np
import TTS as _tts_pkg
print(f'   numpy: {np.__version__}')
print(f'   TTS: {_tts_pkg.__version__}')
print(f'   PyTorch: {torch.__version__}')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# [STEP 5/9] AUTO-FIX: Detect and fix audio paths (FIX BUG #2, #5)
# ══════════════════════════════════════════════════════════════════════════════
print('\n[STEP 5/9] Auto-fixing audio paths and selecting reference audio...')

import soundfile as sf
from typing import Dict, List, Tuple

def auto_fix_paths(
    audio_base_path: str,
    train_manifest_path: str,
    test_manifest_path: str,
    output_dir: str = '/kaggle/working'
) -> Tuple[str, str, int, int]:
    """
    Auto-fix audio paths in manifests.
    
    Returns:
        (fixed_train_path, fixed_test_path, train_count, test_count)
    """
    print("="*80)
    print("[AUTO-FIX] Finding audio files and fixing manifest paths...")
    print("="*80)
    
    # Step 1: Find all .wav files
    print(f"\n[1/4] Scanning for .wav files in: {audio_base_path}")
    wav_files = {}
    for root, dirs, files in os.walk(audio_base_path):
        for file in files:
            if file.endswith('.wav'):
                full_path = os.path.join(root, file)
                # Use basename without extension as key
                key = os.path.splitext(file)[0]
                wav_files[key] = full_path
    
    if not wav_files:
        raise RuntimeError(f"❌ No .wav files found in {audio_base_path}")
    
    print(f"✅ Found {len(wav_files):,} .wav files")
    print(f"   Example: {list(wav_files.values())[0]}")
    
    # Step 2: Fix train manifest
    print(f"\n[2/4] Fixing train manifest...")
    fixed_train_path = os.path.join(output_dir, 'train_manifest_fixed.jsonl')
    train_valid, train_missing = fix_manifest(
        train_manifest_path, fixed_train_path, wav_files
    )
    
    # Step 3: Fix test manifest
    print(f"\n[3/4] Fixing test manifest...")
    fixed_test_path = os.path.join(output_dir, 'test_manifest_fixed.jsonl')
    test_valid, test_missing = fix_manifest(
        test_manifest_path, fixed_test_path, wav_files
    )
    
    # Step 4: Validate
    print(f"\n[4/4] Validation:")
    print(f"   Train: {train_valid:,} valid, {train_missing:,} missing")
    print(f"   Test:  {test_valid:,} valid, {test_missing:,} missing")
    
    if train_valid == 0:
        raise RuntimeError(
            f"❌ FATAL: No valid training samples!\n"
            f"   Manifest: {train_manifest_path}\n"
            f"   Audio dir: {audio_base_path}\n"
            f"   This means the audio filenames in the manifest don't match\n"
            f"   the actual .wav files found in the dataset."
        )
    
    print(f"\n✅ Manifests fixed successfully!")
    print(f"   Train: {fixed_train_path}")
    print(f"   Test:  {fixed_test_path}")
    
    return fixed_train_path, fixed_test_path, train_valid, test_valid


def fix_manifest(
    src_path: str,
    dst_path: str,
    wav_files: Dict[str, str]
) -> Tuple[int, int]:
    """
    Fix a single manifest file.
    
    Returns:
        (valid_count, missing_count)
    """
    valid = 0
    missing = 0
    
    with open(src_path, 'r', encoding='utf-8') as fin, \
         open(dst_path, 'w', encoding='utf-8') as fout:
        for line in fin:
            line = line.strip()
            if not line:
                continue
            
            try:
                obj = json.loads(line)
            except json.JSONDecodeError:
                missing += 1
                continue
            
            # Get audio path and text
            old_audio = obj.get('audio', '')
            text = obj.get('text', '').strip()
            
            if not old_audio or not text:
                missing += 1
                continue
            
            # Extract basename without extension
            basename = os.path.splitext(os.path.basename(old_audio))[0]
            
            # Look up in wav_files dict
            if basename in wav_files:
                obj['audio'] = wav_files[basename]
                fout.write(json.dumps(obj, ensure_ascii=False) + '\n')
                valid += 1
            else:
                missing += 1
    
    return valid, missing


def select_reference_audio(
    audio_base_path: str,
    target_min: float = 5.0,
    target_max: float = 10.0
) -> str:
    """
    Select the best reference audio file.
    
    Criteria:
    1. Duration between 5-10 seconds (preferred)
    2. Highest sample rate
    3. Falls back to any .wav if no ideal file found
    
    Returns:
        Absolute path to selected reference audio
    """
    print("\n" + "="*80)
    print("[AUTO-SELECT] Finding best reference audio...")
    print("="*80)
    
    # Find all .wav files
    wav_files = []
    for root, dirs, files in os.walk(audio_base_path):
        for file in files:
            if file.endswith('.wav'):
                wav_files.append(os.path.join(root, file))
    
    if not wav_files:
        raise RuntimeError(f"❌ No .wav files found in {audio_base_path}")
    
    print(f"Scanning {len(wav_files)} audio files...")
    
    # Evaluate candidates
    candidates = []
    for wav_path in wav_files[:200]:  # Check first 200 files
        try:
            info = sf.info(wav_path)
            duration = info.frames / info.samplerate
            
            # Score based on duration and sample rate
            if target_min <= duration <= target_max:
                duration_score = 100
            elif duration < target_min:
                duration_score = 50 * (duration / target_min)
            else:
                duration_score = 50 * (target_max / duration)
            
            sr_score = info.samplerate / 1000
            total_score = duration_score + sr_score
            
            candidates.append({
                'path': wav_path,
                'duration': duration,
                'sample_rate': info.samplerate,
                'score': total_score
            })
        except Exception:
            continue
    
    if not candidates:
        # Fallback: use first .wav file
        print("⚠️  Could not analyze audio files, using first .wav")
        return wav_files[0]
    
    # Sort by score and pick best
    candidates.sort(key=lambda x: x['score'], reverse=True)
    best = candidates[0]
    
    print(f"✅ Selected reference audio:")
    print(f"   File: {os.path.basename(best['path'])}")
    print(f"   Duration: {best['duration']:.2f}s")
    print(f"   Sample rate: {best['sample_rate']} Hz")
    
    return best['path']

# Define paths
AUDIO_BASE = '/kaggle/input/datasets/tinthnhphm21022004/data-speech-to-text'
MANIFEST_BASE = '/kaggle/input/datasets/thanhphamtien2102224/weight-phowhisper'
TRAIN_MANIFEST_SRC = f'{MANIFEST_BASE}/train_full_manifest.jsonl'
TEST_MANIFEST_SRC = f'{MANIFEST_BASE}/test_manifest.jsonl'

# Validate dataset paths exist
if not os.path.exists(AUDIO_BASE):
    raise RuntimeError(
        f'❌ Audio dataset not found: {AUDIO_BASE}\n'
        f'   Please add dataset: tinthnhphm21022004/data-speech-to-text'
    )

if not os.path.exists(TRAIN_MANIFEST_SRC):
    raise RuntimeError(
        f'❌ Train manifest not found: {TRAIN_MANIFEST_SRC}\n'
        f'   Please add dataset: thanhphamtien2102224/weight-phowhisper'
    )

# FIX BUG #2: Auto-fix audio paths
TRAIN_MANIFEST, TEST_MANIFEST, train_count, test_count = auto_fix_paths(
    audio_base_path=AUDIO_BASE,
    train_manifest_path=TRAIN_MANIFEST_SRC,
    test_manifest_path=TEST_MANIFEST_SRC,
    output_dir='/kaggle/working'
)
print('✅ [BUG #2 FIXED] Audio paths corrected')

# FIX BUG #5: Auto-select reference audio
REFERENCE_AUDIO = select_reference_audio(AUDIO_BASE)
print('✅ [BUG #5 FIXED] Reference audio selected')

# Summary
print('\n' + '='*80)
print('📊 AUTO-FIX SUMMARY:')
print('='*80)
print(f'✅ Train samples:    {train_count:,}')
print(f'✅ Val samples:      {test_count:,}')
print(f'✅ Reference audio:  {os.path.basename(REFERENCE_AUDIO)}')
print(f'✅ Ready to train!')
print('='*80)

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# [STEP 6/9] Configure training (FIX BUG #3, #4)
# ══════════════════════════════════════════════════════════════════════════════
print('\n[STEP 6/9] Configuring training...')

from xtts_finetune.config import TrainingConfig
from xtts_finetune.utils import get_logger, set_seed

# FIX BUG #3 & #4: Proper config with T4 optimizations
config = TrainingConfig(
    # Model
    hf_repo_id='coqui/XTTS-v2',
    base_model_dir='/kaggle/working/base_model',
    
    # Data (using fixed manifests)
    train_manifest=TRAIN_MANIFEST,
    val_manifest=TEST_MANIFEST,
    reference_audio=REFERENCE_AUDIO,
    audio_root_remap=None,  # Already fixed in manifests
    
    # FIX BUG #4: T4 OOM prevention
    batch_size=2,
    grad_accum_steps=8,
    patch_size=2000,
    use_fp16=True,
    gradient_checkpointing=True,
    freeze_encoder=True,
    num_workers=1,
    
    # Training
    epochs_per_patch=1,
    learning_rate=2e-5,
    eval_every_n_steps=500,
    save_every_n_steps=500,
    
    # Output
    output_dir='/kaggle/working/output',
    checkpoint_dir='/kaggle/working/output/checkpoints',
    sample_dir='/kaggle/working/output/samples',
    log_dir='/kaggle/working/output/logs',
    
    # Misc
    seed=42,
    speaker_mode='single',
    zip_checkpoints=False,
)

logger = get_logger('kaggle_training', config.log_dir)
set_seed(42)

print('✅ [BUG #3 FIXED] audio_root_remap configured')
print('✅ [BUG #4 FIXED] T4 OOM prevention enabled')
print(f'   Batch size: {config.batch_size}')
print(f'   Gradient accumulation: {config.grad_accum_steps}')
print(f'   Effective batch: {config.batch_size * config.grad_accum_steps}')
print(f'   FP16: {config.use_fp16}')
print(f'   Gradient checkpointing: {config.gradient_checkpointing}')
print(f'   Freeze encoder: {config.freeze_encoder}')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# [STEP 7/9] Load and validate dataset
# ══════════════════════════════════════════════════════════════════════════════
print('\n[STEP 7/9] Loading and validating dataset...')

from xtts_finetune.dataset import load_manifest, validate_and_filter

train_raw = load_manifest(config.train_manifest, logger, config.audio_root_remap)
val_raw = load_manifest(config.val_manifest, logger, config.audio_root_remap)

train_samples = validate_and_filter(train_raw, config, logger)
val_samples = validate_and_filter(val_raw, config, logger)

print(f'✅ Train: {len(train_samples):,} valid samples')
print(f'✅ Val: {len(val_samples):,} valid samples')

if len(train_samples) == 0:
    raise RuntimeError('❌ No valid training samples after validation!')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# [STEP 8/9] Download model and prepare for training
# ══════════════════════════════════════════════════════════════════════════════
print('\n[STEP 8/9] Downloading base model and preparing...')

from xtts_finetune.model_loader import (
    download_base_model,
    load_xtts_model,
    configure_trainable_params,
    enable_gradient_checkpointing,
    extract_speaker_embedding,
)
from xtts_finetune.utils import log_gpu_memory, free_memory

# Download base model
download_base_model(config, logger)
print('✅ Base model downloaded')

# Load model
model, xtts_config = load_xtts_model(config, logger)
model = configure_trainable_params(model, config, logger)
enable_gradient_checkpointing(model, logger)
log_gpu_memory(logger, 'after model load')

# Extract speaker embedding
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
speaker_embedding = extract_speaker_embedding(
    model, xtts_config, config.reference_audio, device, logger
)

if speaker_embedding is not None:
    print(f'✅ Speaker embedding: {speaker_embedding.shape}')
else:
    print('⚠️  Using default speaker embedding')

# Clear memory
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.synchronize()
log_gpu_memory(logger, 'after cleanup')
print('✅ Memory cleared, ready for training')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# [STEP 9/9] Train
# ══════════════════════════════════════════════════════════════════════════════
print('\n[STEP 9/9] Starting training...')
print('='*80)

from xtts_finetune.trainer import XTTSTrainer

trainer = XTTSTrainer(
    model=model,
    xtts_config=xtts_config,
    config=config,
    speaker_embedding=speaker_embedding,
)

try:
    final_metrics = trainer.train(train_samples, val_samples)
    
    print('\n' + '='*80)
    print('🎉 TRAINING COMPLETED SUCCESSFULLY!')
    print('='*80)
    print(f'Best validation loss: {trainer.best_val_loss:.4f}')
    print(f'Total steps: {trainer.global_step}')
    print(f'\n📂 Output locations:')
    print(f'   Checkpoints: {config.checkpoint_dir}')
    print(f'   Samples: {config.sample_dir}')
    print(f'   Logs: {config.log_dir}')
    
except KeyboardInterrupt:
    print('\n⚠️  Training interrupted by user')
    print(f'Partial results saved to: {config.output_dir}')
    
except Exception as e:
    print('\n' + '='*80)
    print('❌ TRAINING FAILED')
    print('='*80)
    print(f'Error: {type(e).__name__}: {e}')
    
    import traceback
    traceback.print_exc()
    
    # Save debug info
    debug_file = f'{config.output_dir}/error_debug.txt'
    os.makedirs(os.path.dirname(debug_file), exist_ok=True)
    with open(debug_file, 'w') as f:
        f.write(f'Error: {type(e).__name__}: {e}\n\n')
        f.write('Traceback:\n')
        traceback.print_exc(file=f)
    print(f'\n💾 Debug info saved to: {debug_file}')
    raise

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# FINAL SUMMARY
# ══════════════════════════════════════════════════════════════════════════════
print('\n' + '='*80)
print('✅ ALL STEPS COMPLETED!')
print('='*80)
print('\n📊 Training Summary:')
print(f'   Total samples trained: {len(train_samples):,}')
print(f'   Validation samples: {len(val_samples):,}')
print(f'   Total steps: {trainer.global_step}')
print(f'   Best val loss: {trainer.best_val_loss:.4f}')
print('\n📂 Output locations:')
print(f'   Checkpoints: {config.checkpoint_dir}')
print(f'   Samples: {config.sample_dir}')
print(f'   Logs: {config.log_dir}')
print('\n🎉 Training pipeline completed successfully!')
print('='*80)